# RAG at Scale — Complete Demo

**How to run (pick one):**

| Method | Command | Kernel to select |
|---|---|---|
| **Local** (recommended now) | Open this notebook in VS Code / Jupyter | `rag-at-scale` (Python 3.11 from this repo's `venv`) |
| **Docker** | `run_demo.bat` → open http://localhost:8888?token=ragdemo | pre-selected inside container |

**Topics:** Spark · Distributed Embeddings · Hybrid Search · Rerank · FastAPI · Observability · AWS Bedrock · Docker Deploy

**Story hook:** the demo corpus mixes one real RAG trend brief with several fictional failure incidents, so students can clearly see when chunking, hybrid retrieval, and reranking recover the right narrative.

---

In [ ]:
import sys, os, json, time, glob
from pathlib import Path
import numpy as np
import pandas as pd

# Tell Spark's JVM to use THIS Python for worker processes
os.environ['PYSPARK_PYTHON']        = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Support both Docker (/home/jovyan/work) and local execution (cwd)
_docker_root = Path('/home/jovyan/work')
PROJECT_ROOT = _docker_root if _docker_root.exists() else Path('.').resolve()
SRC = str(PROJECT_ROOT / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

# Windows only: winutils.exe needed by Hadoop filesystem layer
if os.name == 'nt':
    os.environ.setdefault('HADOOP_HOME', r'C:\hadoop')
    os.environ['PATH'] = os.environ['HADOOP_HOME'] + r'\bin;' + os.environ.get('PATH', '')

import pyspark
print(f'✅ Python      : {sys.version.split()[0]}')
print(f'✅ PySpark     : {pyspark.__version__}')
print(f'✅ NumPy       : {np.__version__}')
print(f'✅ Pandas      : {pd.__version__}')
print(f'✅ Project root: {PROJECT_ROOT}')

---
## Section 1 — Spark Setup

**Key concepts:** Driver · Executors · Lazy evaluation · Catalyst optimizer

**Cluster tuning knobs to show students:**
| Config | Effect |
|---|---|
| `spark.driver.memory` | RAM for the driver process |
| `spark.executor.memory` | RAM per executor |
| `spark.sql.adaptive.enabled` | Auto-optimise join strategy at runtime |
| `spark.default.parallelism` | Default partitions for RDD ops |

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName('RAG_at_Scale_Demo')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .config('spark.executor.memory', '2g')
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

print(f'✅ Spark {spark.version}   UI → http://localhost:4040')

# Quick sanity check
df = spark.createDataFrame([('Alice', 25), ('Bob', 30), ('Charlie', 35)], ['name', 'age'])
df.show()
print(f'Rows: {df.count()}')

In [ ]:
# ── Lazy evaluation demo — nothing executes until .show() / .count() ──────
# Use the real raw docs for the demo (loaded from data/raw/*.txt)
raw_files = list((PROJECT_ROOT / 'data' / 'raw').glob('*.txt'))
demo_records = [(p.stem, p.read_text('utf-8').strip()) for p in raw_files]
demo_df = spark.createDataFrame(demo_records, ['doc_id', 'content'])

pipeline = (
    demo_df
    .withColumn('word_count', size(split(col('content'), ' ')))
    .withColumn('char_count', length(col('content')))
    .groupBy('doc_id')
    .agg(
        avg('word_count').alias('avg_words'),
        avg('char_count').alias('avg_chars')
    )
)

print('--- Logical plan (not yet executed) ---')
pipeline.explain()
print('--- Triggering execution now ---')
pipeline.show(truncate=False)

---
## Section 2 — Document Processing

**Key concepts:** Native Spark functions vs Python UDFs · Chunking with overlap · Data quality

In [ ]:
# Load the sample .txt files
raw_files = sorted((PROJECT_ROOT / 'data' / 'raw').glob('*.txt'))
print(f'Found {len(raw_files)} files')

records = [(p.stem, p.read_text('utf-8').strip()) for p in raw_files]
raw_df = spark.createDataFrame(records, ['doc_id', 'content'])
raw_df.show(truncate=80)

from IPython.display import Markdown, display

story_sections = [
    '## Story Corpus Preview',
    'This corpus intentionally mixes one real trend brief with fictional RAG incident reports so retrieval mistakes and reranking wins are easy to explain live.'
]
for doc_id, content in records:
    preview = content.replace('\n', ' ').strip()
    preview = preview[:260] + ('...' if len(preview) > 260 else '')
    story_sections.append(f"### `{doc_id}`\n{preview}")

display(Markdown('\n\n'.join(story_sections)))

In [ ]:
# ── TEACHING POINT ─────────────────────────────────────────────────────────
# Use NATIVE Spark SQL functions whenever possible:
#   ✅ lower(), trim(), regexp_replace(), size(), split()  → JVM, no pickle
#   ⚠️  Python UDF                                         → serialised via cloudpickle
# ──────────────────────────────────────────────────────────────────────────
clean_df = (
    raw_df
    .withColumn('clean', trim(regexp_replace(lower(col('content')), r'\s+', ' ')))
    .withColumn('word_count', size(split(col('clean'), ' ')))
    .withColumn('sentence_count', size(split(col('clean'), r'\.\s+')))
)
clean_df.select('doc_id', 'word_count', 'sentence_count', 'clean').show(truncate=70)

In [ ]:
# ── Document chunking ────────────────────────────────────────────
# NOTE: Do NOT use max() here — pyspark.sql.functions.* shadows the builtin.
def chunk_words(text, chunk_size=80, overlap=15):
    if not text: return []
    words = text.split()
    step = chunk_size - overlap if chunk_size > overlap else 1   # avoids builtin max()
    return [" ".join(words[i:i+chunk_size])
            for i in range(0, len(words), step)
            if len(words[i:i+chunk_size]) >= chunk_size // 2]

# ── TEACHING POINT: production pattern would be a Pandas UDF ────────────────
#   chunk_udf = udf(chunk_words, ArrayType(StringType()))
#   chunks_df = (clean_df
#       .withColumn("chunks", chunk_udf("clean"))
#       .withColumn("chunk", explode("chunks")))
#
# For this demo (5 docs) we chunk on the driver to avoid Spark UDF worker
# socket instability under nbconvert / batch execution on Windows.
# ──────────────────────────────────────────────────────────────────────────
chunk_records = []
for row in clean_df.collect():
    for i, chunk in enumerate(chunk_words(row["clean"])):
        chunk_id = f"{row['doc_id']}_{i}"
        chunk_records.append((chunk_id, row["doc_id"], chunk))

from pyspark.sql.types import StructType, StructField, StringType as ST
chunk_schema = StructType([
    StructField("chunk_id", ST()),
    StructField("doc_id",   ST()),
    StructField("chunk",    ST()),
])
chunks_df = spark.createDataFrame(chunk_records, chunk_schema)
print(f"Total chunks: {chunks_df.count()}")
chunks_df.show(5, truncate=65)

---
## Section 3 — Distributed Embeddings

**Key concepts:** Pandas UDF · Per-executor model caching · GPU scheduling · Batch size tuning

**GPU cluster config (production):**
```python
spark.config('spark.executor.resource.gpu.amount', '1')
spark.config('spark.task.resource.gpu.amount', '1')
spark.config('spark.executor.cores', '4')  # 4 CPU cores per GPU executor
```

In [ ]:
import torch
gpu = torch.cuda.is_available()
device = 'cuda' if gpu else 'cpu'
print(f'GPU available : {gpu}')
if gpu:
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}  {p.total_memory/1024**3:.1f} GB')
else:
    print('  Using CPU — embeddings will be slower but functionally identical')
print(f'Device: {device}')

In [ ]:
from embeddings.spark_embedder import SparkEmbedder

embedder = SparkEmbedder()          # uses all-MiniLM-L6-v2 (384-dim)

# Driver-side test
test_emb = embedder.embed_batch(['Machine learning is powerful.'])[0]
print(f'Model : {embedder.model_name}')
print(f'Dim   : {len(test_emb)}')
print(f'Sample: {test_emb[:5]}')

In [ ]:
# ── TEACHING POINT — Pandas UDF for distributed embedding ─────────────────
# In production (hundreds of GB of docs) you'd run this on a cluster:
#
#   embedded_df = chunks_df.withColumn('embedding', embedding_udf('chunk'))
#
# Key insight: capture only the model NAME (a string) in the closure.
# Each executor worker loads + caches the model once per process.
# ──────────────────────────────────────────────────────────────────────────
embedding_udf = embedder.get_embedding_udf()
print('Pandas UDF defined — model:', embedder.model_name)
print()

# ── Demo mode: embed on the driver (5 docs = no need for a cluster) ────────
# Collect text to driver → batched encode → rebuild as Spark DataFrame.
# Identical schema to the Pandas UDF path; all downstream cells just work.
n_total = chunks_df.count()
print(f'Embedding {n_total} chunks on driver ...')
t0 = time.time()

rows    = chunks_df.collect()
texts   = [r.chunk for r in rows]
vectors = embedder.embed_batch(texts)

elapsed = time.time() - t0
print(f'✅ {len(vectors)} chunks embedded in {elapsed:.1f}s  ({len(vectors)/elapsed:.1f} chunks/sec)')
print(f'   Embedding dim : {len(vectors[0])}')

# Rebuild as Spark DataFrame
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, FloatType
embed_schema = StructType([
    StructField('chunk_id',  StringType()),
    StructField('doc_id',    StringType()),
    StructField('chunk',     StringType()),
    StructField('embedding', ArrayType(FloatType())),
])
embed_records = [(r.chunk_id, r.doc_id, r.chunk, [float(x) for x in v])
                 for r, v in zip(rows, vectors)]
embedded_df = spark.createDataFrame(embed_records, embed_schema)
embedded_df.cache()
embedded_df.select('chunk_id', 'chunk', 'embedding').show(3, truncate=50)

In [ ]:
EMBED_PATH = str(PROJECT_ROOT / 'data' / 'embeddings' / 'chunk_embeddings.parquet')
(PROJECT_ROOT / 'data' / 'embeddings').mkdir(parents=True, exist_ok=True)
embedded_df.write.mode('overwrite').parquet(EMBED_PATH)
print(f'💾 Embeddings saved to {EMBED_PATH}')

---
## Section 4 — Hybrid Search (Dense + BM25)

**Key concepts:** Cosine similarity · BM25 · Min-max normalisation · Alpha fusion

```
final_score = α × dense_score + (1-α) × bm25_score
  α = 1.0 → pure semantic   |   α = 0.0 → pure keyword
```

In [ ]:
from IPython.display import Markdown, display
from retrieval.hybrid_search import HybridSearch
from retrieval.reranker import Reranker

search_engine = HybridSearch()
reranker = Reranker()

# Load embeddings and build index
rows = spark.read.parquet(EMBED_PATH).collect()
docs       = [{'id': r.chunk_id, 'content': r.chunk} for r in rows]
embeddings = [r.embedding for r in rows]

search_engine.index_documents(docs, embeddings)
print(f'✅ Indexed {len(docs)} chunks')

doc_titles = {
    'sample_doc_1': 'Real brief: GraphRAG and agentic AI',
    'sample_doc_2': 'Incident: Labyrinthine-Drift in the Eon-Registry',
    'sample_doc_3': 'Incident: Veritas-9 legal hallucinations',
    'sample_doc_4': 'Incident: Solstice-Core collapse and Aether-Reranker failure',
    'sample_doc_5': 'Incident: Void-Leakage in Chronos-Deep',
}
chunk_lookup = {doc['id']: doc['content'] for doc in docs}

def source_doc_id(chunk_id: str) -> str:
    return chunk_id.rsplit('_', 1)[0] if '_' in chunk_id else chunk_id

def source_label(chunk_id: str) -> str:
    return doc_titles.get(source_doc_id(chunk_id), source_doc_id(chunk_id))

def show_hits(results, score_key: str = 'score', title: str = 'Top retrieved chunks'):
    lines = [f'### {title}']
    for i, hit in enumerate(results, 1):
        chunk_id = hit['doc_id']
        snippet = chunk_lookup.get(chunk_id, '').replace('\n', ' ').strip()
        snippet = snippet[:180] + ('...' if len(snippet) > 180 else '')
        score = hit.get(score_key, hit.get('score', 0.0))
        lines.append(
            f"{i}. **{source_label(chunk_id)}**  \\n"
            f"   Chunk: `{chunk_id}` | {score_key}={score:.3f}  \\n"
            f"   {snippet}"
        )
    display(Markdown('\n\n'.join(lines)))

In [ ]:
# ── Alpha sweep — shows how fusion weight changes results ──────────────────
query = 'In the Veritas-9 incident, why did the team add a cross-encoder before generation?'
q_emb = embedder.embed_batch([query])[0]

print(f'Query: "{query}"\n')
print(f'{"Mode":<22}  {"Score":>6}  Top source')
print('-' * 90)
for alpha in [0.0, 0.3, 0.5, 0.7, 1.0]:
    results = search_engine.search(query, q_emb, top_k=3, alpha=alpha)
    top = results[0] if results else {}
    label = 'pure keyword' if alpha == 0 else ('pure semantic' if alpha == 1 else f'hybrid α={alpha}')
    source = source_label(top.get('doc_id', '')) if top else '(none)'
    print(f'{label:<22}  {top.get("score", 0):>6.3f}  {source}')

best_results = search_engine.search(query, q_emb, top_k=4, alpha=0.6)
show_hits(best_results, title='Hybrid retrieval preview for the Veritas-9 question')

---
## Section 5 — Cross-Encoder Reranking

**Why rerank?** Bi-encoder retrieval is fast but approximate. Cross-encoders see query + document jointly → better relevance, but slower. Use for top-K re-scoring only.

In [ ]:
reranker.load_model()

query = 'In the Veritas-9 incident, why did the team add a cross-encoder before generation?'
q_emb = embedder.embed_batch([query])[0]

# Retrieve more candidates, then rerank
candidates = search_engine.search(query, q_emb, top_k=8, alpha=0.6)
docs_for_rerank = [
    {'content': next((d['content'] for d in docs if d['id'] == r['doc_id']), ''),
     'doc_id': r['doc_id'],
     'score': r['score']}
    for r in candidates
]

print(f'Query: "{query}"\n')
show_hits(candidates[:4], title='Before reranking')

reranked = reranker.rerank(query, docs_for_rerank, top_k=3)
show_hits(reranked, score_key='rerank_score', title='After reranking')

---
## Section 6 — FastAPI RAG Service

In [ ]:
from service.app import app
from service.handlers import rag_service
from fastapi.testclient import TestClient

# Wire up the service with the indexed data
rag_service.initialize_search(docs, embeddings)
client = TestClient(app)

service_query = 'In the Veritas-9 incident, why did the team add a cross-encoder before generation?'

# Health
health = client.get('/health').json()
print('Health:', health['status'], '|', health['services'])

# Search
resp = client.post('/search', json={'query': service_query, 'top_k': 3, 'rerank': True}).json()
print(f'\nService query: {resp["query"]}')
print(f'Search: {resp["search_time"]*1000:.0f}ms  Rerank: {(resp.get("rerank_time") or 0)*1000:.0f}ms')
service_hits = [
    {
        'doc_id': r['doc_id'],
        'content': r['content'],
        'score': r['score'],
        'rerank_score': r.get('rerank_score'),
    }
    for r in resp['results']
]
show_hits(service_hits, score_key='rerank_score', title='FastAPI /search response')

In [ ]:
print('Live service options:')
print('  Local  -> uvicorn src.service.app:app --host 0.0.0.0 --port 8000 --reload')
print('  Docker -> docker compose -f docker-compose.yml up --build')
print('  Docs   -> http://localhost:8000/docs')

---
## Section 7 — Observability: Logs · Traces · Metrics

**Stack:**
- **Structured logging** → JSON lines consumed by CloudWatch / ELK
- **OpenTelemetry** → traces to Jaeger / CloudWatch X-Ray
- **Prometheus** → metrics scraped every 15s, visualised in Grafana
- **Auto-scale trigger** → `rag_searches_performed_total` rate → ECS task count

In [ ]:
from observability.logging_config import StructuredLogger, log_request, log_search_query

StructuredLogger.setup_logging(log_level='INFO', json_format=True)
logger = StructuredLogger.get_logger('demo')

logger.info('RAG demo started', extra={'section': 'observability', 'student': 'demo'})
log_request('req-001', 'POST', '/search', query='Veritas-9 cross-encoder question')
log_search_query('In the Veritas-9 incident, why did the team add a cross-encoder before generation?', 3, 0.042)
print('✅ Structured JSON logs emitted — see console output above')

In [ ]:
from opentelemetry import trace
from observability.tracing import tracing_manager, trace_search_query, trace_embedding_generation

tracing_manager.get_tracer()  # setup if needed, otherwise reuse the existing provider

with trace_search_query(query_length=20):
    time.sleep(0.04)

with trace_embedding_generation(doc_count=50):
    time.sleep(0.02)

trace.get_tracer_provider().force_flush()
time.sleep(0.2)

print('✅ Spans emitted to console in this section')
print('   In production → set OTLP_ENDPOINT to send to Jaeger or CloudWatch X-Ray')
print('   We flush here so later sections stay focused on retrieval and generation output.')

In [ ]:
import threading
from observability.metrics import metrics

# Start Prometheus endpoint in background
threading.Thread(target=lambda: metrics.start_server(port=8002), daemon=True).start()
time.sleep(0.3)

# Simulate some activity
metrics.searches_performed.inc(10)
metrics.embeddings_generated.inc(50)
metrics.index_size.set(len(docs))
with metrics.search_duration.time(): time.sleep(0.02)

import requests
raw = requests.get('http://localhost:8002/metrics').text
interesting = {}
for line in raw.splitlines():
    if line.startswith('rag_searches_performed_total'):
        interesting['searches_performed_total'] = line.split()[-1]
    elif line.startswith('rag_embeddings_generated_total'):
        interesting['embeddings_generated_total'] = line.split()[-1]
    elif line.startswith('rag_index_size'):
        interesting['index_size'] = line.split()[-1]

for key, value in interesting.items():
    print(f'{key:>28}: {value}')

print('\n✅ Prometheus metrics live at http://localhost:8002/metrics')
print('   Start Grafana: docker compose -f docker-compose-demo.yml --profile observability up')

---
## Section 7B — RAG Generation with AWS Bedrock (Claude)

This is the **G** in RAG. Retrieved chunks become the grounded context for the LLM.

In [ ]:
import boto3, json as _json

bedrock = boto3.client('bedrock-runtime', region_name=os.environ.get('AWS_REGION', 'us-east-1'))

# Verify AWS credentials
try:
    sts = boto3.client('sts').get_caller_identity()
    print(f'✅ AWS Account: {sts["Account"]}   ARN: {sts["Arn"]}')
except Exception as e:
    print(f'⚠️  AWS not configured: {e}')
    print('   Run: aws configure')

In [ ]:
MODEL_ID = os.environ.get('BEDROCK_MODEL_ID', 'anthropic.claude-haiku-4-5-20251001-v1:0')
print(f'Using Bedrock model/profile: {MODEL_ID}')

def rag_answer(query: str, top_k: int = 3) -> str:
    q_emb = embedder.embed_batch([query])[0]
    hits = search_engine.search(query, q_emb, top_k=top_k * 2, alpha=0.6)
    pool = [{'content': next((d['content'] for d in docs if d['id'] == h['doc_id']), ''),
             'doc_id': h['doc_id']}
            for h in hits if any(d['id'] == h['doc_id'] for d in docs)]
    ranked = reranker.rerank(query, pool, top_k=top_k) if pool else []

    context = '\n\n'.join(f'[{i+1}] {d["content"]}' for i, d in enumerate(ranked)) or '(no context)'
    body = _json.dumps({
        'anthropic_version': 'bedrock-2023-05-31',
        'max_tokens': 400,
        'system': 'Answer only from the provided context. Be concise.',
        'messages': [{'role': 'user',
                      'content': f'Context:\n{context}\n\nQuestion: {query}'}]
    })
    resp = bedrock.invoke_model(modelId=MODEL_ID, contentType='application/json',
                                accept='application/json', body=body)
    return _json.loads(resp['body'].read())['content'][0]['text']

print('These questions deliberately move across the story corpus: one real trend brief and two fictional RAG incidents.')

# Queries grounded in the actual doc content
for q in [
    'What is GraphRAG and how does it improve on standard RAG?',
    'In the Veritas-9 incident, why did the team add a cross-encoder before generation?',
    'What caused the Void-Leakage in the Chronos-Deep repository and how was it fixed?',
]:
    print(f'\n❓ {q}')
    try:
        print(f'🤖 {rag_answer(q)}')
    except Exception as e:
        msg = str(e)
        if 'inference profile' in msg.lower():
            print('⚠️  This account needs a Bedrock inference profile for the selected model.')
            print('   Set BEDROCK_MODEL_ID to your profile ID or ARN, then rerun this cell.')
        else:
            print(f'⚠️  {msg}')

---
## Section 8 — Docker & AWS Deploy

Use one of these live-demo paths:

- **Local notebook + local API** for the fastest classroom flow
- **Docker notebook** via `docker compose -f docker-compose-demo.yml up`
- **Docker API** via `docker compose -f docker-compose.yml up --build`

### Production deployment flow
```
1. build image     → docker build -f docker/Dockerfile -t rag-service .
2. push to ECR     → bash scripts/deploy_aws.sh
3. rolling deploy  → aws ecs update-service --force-new-deployment
4. autoscale       → HPA on rag_searches_performed_total (Prometheus → ECS capacity provider)
```

In [ ]:
import shutil
import subprocess

checks = [
    (['docker', '--version'], 'Docker'),
    (['docker', 'compose', 'version'], 'Docker Compose'),
    (['aws', '--version'], 'AWS CLI'),
    (['aws', 'sts', 'get-caller-identity', '--query', 'Account', '--output', 'text'], 'AWS Account'),
]

for cmd, label in checks:
    binary = shutil.which(cmd[0])
    if not binary:
        print(f'❌ {label}: {cmd[0]} not installed')
        continue

    r = subprocess.run(cmd, capture_output=True, text=True)
    status = '✅' if r.returncode == 0 else '❌'
    print(f'{status} {label}: {(r.stdout or r.stderr).strip()[:100]}')

print('\nDeployment commands:')
print('  docker compose -f docker-compose.yml up --build   # local API in Docker')
print('  docker compose -f docker-compose-demo.yml up      # Jupyter + Spark in Docker')
print('  bash scripts/deploy_aws.sh                        # build → ECR push → ECS rolling deploy')

---
## Section 9 — Autoscale with Kubernetes HPA

The [kubernetes/hpa.yaml](kubernetes/hpa.yaml) scales ECS tasks / K8s pods when search throughput rises.

**Cluster tuning recap for students:**

| Knob | Recommendation |
|---|---|
| Spark partitions | `2-4 × number of CPU cores` |
| Embedding batch size | `32` (CPU) / `128–256` (GPU) |
| UDF model cache | module-level dict, keyed by model name |
| HPA metric | `rag_searches_performed_total` rate |
| Docker memory limit | `driver_mem + executor_mem + 1g overhead` |

---
**🎓 Demo complete.** You ran the full RAG at Scale pipeline:

> **Spark** chunk → **Pandas UDF** embed → **Hybrid search** → **Cross-encoder** rerank → **FastAPI** serve → **Bedrock Claude** generate → **Prometheus/Grafana** observe → **ECR/ECS** deploy